<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 Amigos con Gustos Parecidos: Cómo Adivinan lo que te Gustará 🎬
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 09
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/09%20-%20Casos%20de%20Estudio%20y%20Aplicaciones/Para%20Dummies/03_Recomendacion_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 03 del Módulo 09 (Sistemas de recomendación)](../03_Sistemas_de_Recomendacion.ipynb). Cuando una plataforma te dice *«te puede gustar esta película»*, no sabe de cine: **se fija en quién tiene gustos parecidos a los tuyos** y en qué le gustó **a esas personas**. Es lo mismo que haces al pedirle una recomendación **a tu amigo que siempre coincide contigo**.

Al terminar podrás explicar:
1. Por qué la **tabla de gustos** casi siempre está **casi vacía** y qué significa «adivinar» una casilla.
2. Cómo se recomienda **preguntando a los más parecidos** (filtrado colaborativo).
3. Qué son los **gustos ocultos** (factorización de matrices) y qué pasa con **alguien nuevo** (arranque en frío).

> 🔗 Si luego quieres ver las versiones **desde cero y verificadas** (vecindarios, factorización con SGD y ALS, métricas de ranking, popularidad y burbuja de filtro): [cuaderno estándar](../03_Sistemas_de_Recomendacion.ipynb). Antes de esto: [contar palabras como cartas](02_NLP_Dummies.ipynb). Para cerrar el módulo: [el triage de urgencias](00_Casos_Estudio_Dummies.ipynb).

---
## 1. La tabla de gustos casi vacía 🎞️

Cinco amigos han puesto de **1 a 5 estrellas** a seis películas, pero **nadie ha visto todas**. Cada casilla vacía (**¿?**) es **una película que esa persona no ha visto**, y el trabajo de un recomendador es **adivinar qué nota pondría**, para ofrecerle justo lo que más le gustaría. En una plataforma real esta tabla tiene **millones** de filas y de columnas, y **más del 95 % de las casillas vacías**. Los datos de aquí son **inventados**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

nan = np.nan
peliculas = ["Acción 1", "Acción 2", "Romance 1", "Romance 2", "Comedia", "Terror"]
amigos = ["Ana", "Beto", "Carla", "Diego", "Elena"]
# 1-5 estrellas; nan = esa persona NO ha visto esa película (la tabla casi siempre esta muy vacia)
T = np.array([[5, nan, 1, nan, 3, 4],        # Ana
              [5, 4, nan, 2, nan, 5],        # Beto
              [1, nan, 5, 4, 4, nan],        # Carla
              [nan, 2, 5, 5, nan, 1],        # Diego
              [2, 1, nan, 4, 5, 2]])         # Elena
tabla = pd.DataFrame(T, index=amigos, columns=peliculas)
display(tabla)

fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(np.ma.masked_invalid(T), cmap="YlGn", vmin=1, vmax=5)
ax.set_xticks(range(6)); ax.set_xticklabels(peliculas, rotation=20); ax.set_yticks(range(5)); ax.set_yticklabels(amigos)
for (i, j), v in np.ndenumerate(T):
    ax.text(j, i, "¿?" if np.isnan(v) else f"{v:.0f}", ha="center", va="center", fontsize=11, fontweight="bold" if np.isnan(v) else None, color="#dc2626" if np.isnan(v) else "black")
ax.set_title("La tabla de gustos: lo que falta (¿?) es lo que queremos adivinar", fontweight="bold", fontsize=10)
plt.colorbar(im, ax=ax, label="estrellas"); plt.tight_layout(); plt.show()
vacias = np.isnan(T).mean()
print(f"Casillas vacías: {vacias:.0%}. En una plataforma real, lo normal es que esté vacío el 95 % o más.")
assert 0.2 < vacias < 0.4

---
**Lección:** con solo 5 amigos y 6 películas ya falta el 30 % de la tabla. Mira los patrones que **un humano ya ve**: Ana y Beto adoran la acción (5, 5) y no se llevan con el romance; Carla y Diego, al revés. Esos patrones son **la materia prima** de la recomendación.

---

## 2. Pregúntale al amigo más parecido 👫

La idea más antigua y sencilla: para adivinar qué nota pondría **Ana** a «Acción 2» (que no ha visto), **buscamos a los amigos que más se parecen a ella** y miramos **qué nota le pusieron ellos**. «Parecerse» significa **haber puntuado de forma parecida las películas que ambos vieron**: un número entre **1** (gustos iguales) y **−1** (gustos opuestos). Después se hace un **promedio de las notas de los parecidos**, dándole **más peso al más parecido**. A esto se le llama **filtrado colaborativo**: no hace falta saber qué es una película de acción; basta ver **quién coincide contigo**.

In [ ]:
# ¿Que le pareceria a Ana "Acción 2"? Le preguntamos a sus amigos MAS PARECIDOS A ELLA (los que puntuaron igual las peliculas que ambos vieron)
def similitud(a, b):
    """Parecido de gustos: 1 = gustos iguales, -1 = opuestos. Se calcula solo con las peliculas que AMBOS vieron (y restando el promedio de cada uno)."""
    ambos = ~np.isnan(a) & ~np.isnan(b)
    if ambos.sum() < 2:
        return 0.0
    x, y = a[ambos] - np.nanmean(a), b[ambos] - np.nanmean(b)
    return float(x @ y / (np.sqrt(x @ x) * np.sqrt(y @ y) + 1e-12))


ana = T[0]
sims = {amigos[k]: similitud(ana, T[k]) for k in range(1, 5)}
print("Parecido de cada amigo con Ana:", {k: round(v, 2) for k, v in sims.items()})

j = peliculas.index("Acción 2")
votos = [(amigos[k], sims[amigos[k]], T[k, j]) for k in range(1, 5) if not np.isnan(T[k, j]) and sims[amigos[k]] > 0]
print("Amigos parecidos a Ana que SÍ vieron «Acción 2»:", [(n, round(s, 2), int(r)) for n, s, r in votos])
prediccion = sum(s * r for _, s, r in votos) / sum(s for _, s, _ in votos)          # promedio de sus notas, dando mas peso a los mas parecidos
print(f"Predicción para Ana: {prediccion:.1f} estrellas (la nota de Beto, ponderada por lo parecidos que son).")
print("Idea: «a quien se parece a ti le gustó, así que seguramente a ti también». Esto es el FILTRADO COLABORATIVO: no hace falta saber qué es una película de acción; basta mirar quién coincide contigo.")
assert sims["Beto"] > 0 > sims["Carla"] and prediccion > 3.5

---
**Lección:** Beto es **muy parecido** a Ana (≈ 0.93) y Carla, Diego y Elena **no** (valores negativos: gustos opuestos). Como solo Beto vio «Acción 2» (le puso 4), la predicción para Ana es **4 estrellas**. Dos cosas a notar con honestidad: **(1)** con tan pocas coincidencias, un solo amigo decide; en un sistema real se usan **muchos vecinos** y se desconfía de parecidos basados en pocas películas; **(2)** a los muy opuestos (similitud negativa) se les puede **ignorar** o usar «al revés», pero aquí solo usamos a los parecidos.

---

## 3. Los gustos ocultos: descubrir «acción» y «romance» sin que nadie lo diga 🧬

Otra idea, más potente: suponer que cada persona tiene **unos pocos «gustos ocultos»** (por ejemplo, *cuánto le gusta la acción* y *cuánto le gusta el romance*) y que cada película tiene **cuánta acción y cuánto romance contiene**. La nota que alguien le pone a una película sale de **combinar** ambos. Lo notable es que **nadie le dice al modelo** cuáles son los géneros: **los descubre solo** al ajustar números que expliquen las notas conocidas. Esto se llama **factorización de matrices** (es la idea que ganó el famoso concurso de Netflix, y es **parienta de la compresión de imágenes** con SVD del Módulo 05).

Para comprobarlo, inventamos **150 personas y 14 películas** (6 de acción, 6 de romance y 2 neutras), con **45 %** de las notas visibles, y **guardamos un cuarto** de lo visto como **examen sellado**. Comparamos cuatro formas de adivinar.

In [ ]:
# --- Gustos ocultos: ¿y si cada persona tiene "cuanto le gusta la accion" y "cuanto le gusta el romance", sin que nadie lo diga? ---
rng = np.random.default_rng(4)
n_u, n_p = 150, 14
gusto = rng.normal(0, 1, (n_u, 2))                                       # gustos ocultos de cada persona: [accion, romance]
tipo_peli = np.array([0] * 6 + [1] * 6 + [2] * 2)                        # 6 de accion, 6 de romance, 2 "neutras"
rasgos = np.array([[1.0, 0.0] if t == 0 else [0.0, 1.0] if t == 1 else [0.2, 0.2] for t in tipo_peli]) + rng.normal(0, 0.1, (n_p, 2))
real = np.clip(np.round(3.2 + 1.2 * gusto @ rasgos.T + rng.normal(0, 0.4, (n_u, n_p))), 1, 5)
visto = rng.random((n_u, n_p)) < 0.45                                    # solo se ve el 45 % de las calificaciones
prueba = visto & (rng.random((n_u, n_p)) < 0.25)                         # guardamos 1 de cada 4 de lo visto como examen
ent = visto & ~prueba


def aprender(R, M, k=2, lam=1.0, vueltas=30):
    """Mínimos cuadrados alternados: se ajustan los gustos de las personas dejando fijos los rasgos de las peliculas, y luego al revés, una y otra vez."""
    r = np.random.default_rng(0); P, Q = r.normal(0, 0.3, (R.shape[0], k)), r.normal(0, 0.3, (R.shape[1], k))
    mu = R[M].mean()
    for _ in range(vueltas):
        for u in range(R.shape[0]):
            if M[u].any():
                P[u] = np.linalg.solve(Q[M[u]].T @ Q[M[u]] + lam * np.eye(k), Q[M[u]].T @ (R[u, M[u]] - mu))
        for i in range(R.shape[1]):
            if M[:, i].any():
                Q[i] = np.linalg.solve(P[M[:, i]].T @ P[M[:, i]] + lam * np.eye(k), P[M[:, i]].T @ (R[M[:, i], i] - mu))
    return mu + P @ Q.T, Q


pred, Q = aprender(real, ent)
rmse = lambda p: float(np.sqrt(np.mean((np.clip(p, 1, 5)[prueba] - real[prueba]) ** 2)))
media_peli = np.array([real[ent[:, i], i].mean() if ent[:, i].any() else real[ent].mean() for i in range(n_p)])
print(f"Error al adivinar las calificaciones que NO vio el modelo (menor es mejor):")
print(f"  Decir siempre el promedio de todas:         {rmse(np.full((n_u, n_p), real[ent].mean())):.2f}")
print(f"  Decir el promedio de cada película:         {rmse(np.tile(media_peli, (n_u, 1))):.2f}")
print(f"  Descubrir 2 gustos ocultos (factorización): {rmse(pred):.2f}")

fig, ax = plt.subplots(figsize=(7, 5.2))
colores = ["#dc2626", "#2563eb", "#9ca3af"]; nombres = ["acción", "romance", "neutra"]
for t in range(3):
    ax.scatter(Q[tipo_peli == t, 0], Q[tipo_peli == t, 1], s=90, color=colores[t], label=nombres[t])
ax.set_xlabel("Rasgo oculto 1 (descubierto solo)"); ax.set_ylabel("Rasgo oculto 2 (descubierto solo)"); ax.legend()
ax.set_title("Sin que nadie le diga los géneros, el modelo agrupa las películas por gusto", fontweight="bold", fontsize=10)
plt.tight_layout(); plt.show()
assert rmse(pred) < rmse(np.tile(media_peli, (n_u, 1))) - 0.2

---
**Lección:** decir siempre el promedio (o el promedio de cada película) da un error de ≈ 1.1–1.2 estrellas; **descubriendo dos gustos ocultos** baja a ≈ 0.66: **cerca de un 40 % menos**. Y el gráfico muestra que el modelo **agrupó las películas por género sin que nadie se lo dijera**. Una advertencia: este mundo se inventó con **exactamente dos gustos**; en la realidad hay muchos más, mezclados y menos limpios, así que **la ganancia suele ser bastante menor**.

---

## 4. La persona nueva: el problema del «arranque en frío» 🧊

¿Qué recomiendas a **alguien que acaba de llegar** y no ha visto nada? No sabes nada de sus gustos: lo único sensato es recomendarle **lo más popular** (lo que más gente vio), igual para todos. Pero apenas **califique unas pocas películas** ya puedes **personalizar**. Veámoslo con una persona nueva que califica **tres películas de acción** con 5, 5 y 4 estrellas: buscamos a quienes **coinciden con ella** y le recomendamos **lo que ellos aman** y ella aún no vio.

In [ ]:
# --- La persona NUEVA y el exito de taquilla: sin historial, lo unico razonable es recomendar lo mas popular ---
popularidad = ent.sum(0)                                                  # cuantas personas calificaron cada pelicula
top_popular = np.argsort(-popularidad)[:3]
print("Para alguien sin historial: las 3 más vistas son las películas", top_popular.tolist(), "(les damos a todos lo mismo).")

# Una persona nueva que solo califica TRES peliculas de accion (5, 5 y 4 estrellas). Buscamos a quienes coinciden con ella y recomendamos lo que ellos aman y ella no ha visto
nueva = np.full(n_p, np.nan); nueva[[0, 1, 2]] = [5, 5, 4]
visible = np.where(ent, real, np.nan)
def coincidencias(u):
    """+1 por cada pelicula que a ambos les gusto (4-5 estrellas), -1 por cada una que a la otra persona no le gusto (1-2)."""
    return sum(1 if visible[u, i] >= 4 else -1 if visible[u, i] <= 2 else 0 for i in [0, 1, 2] if not np.isnan(visible[u, i]))
vecinos = [u for u in range(n_u) if coincidencias(u) >= 2]                    # los que coinciden en al menos 2 gustos netos
puntaje = np.full(n_p, -np.inf)
for i in range(n_p):
    votos = [visible[u, i] for u in vecinos if not np.isnan(visible[u, i])]
    if np.isnan(nueva[i]) and len(votos) >= 3:                                # exigimos al menos 3 opiniones para fiarnos
        puntaje[i] = np.mean(votos)
rec = np.argsort(-puntaje)[:3]
print("Con solo 3 calificaciones, recomendamos las películas", rec.tolist(), "→ géneros:", [nombres[tipo_peli[i]] for i in rec], "(la persona prefiere ACCIÓN)")
print("Moraleja: al principio solo sabemos lo popular (arranque en frío); con unas pocas calificaciones ya se puede personalizar.")
assert sum(tipo_peli[i] == 0 for i in rec) >= 2

---
**Lección:** sin historial, a **todos** se les ofrecen las mismas tres (las más vistas, de géneros mezclados: **una de acción** entre ellas); con **solo tres calificaciones**, las tres recomendaciones personalizadas son **de acción**. Esto es el **arranque en frío**: al principio solo hay popularidad; pronto, personalización. Y una nota de cautela: **recomendar solo lo popular** hace que lo popular se vuelva **aún más popular** (y lo demás, invisible): en el cuaderno estándar se mide con la *cobertura* del catálogo y la *burbuja de filtro*.

---

## Resumen en una frase 🎯

> **Un recomendador adivina las casillas vacías de una tabla de gustos preguntando a quienes se parecen a ti o descubriendo los gustos ocultos de cada persona y cada película, y cuando no sabe nada de alguien, recurre a lo más popular.**

### Lo que aprendiste hoy:
- ✅ La **tabla de gustos** está casi vacía; **recomendar es adivinar las casillas vacías** (la nota que pondría cada persona).
- ✅ **Filtrado colaborativo:** adivinar con las notas de **los más parecidos** (Beto → Ana: 4 estrellas).
- ✅ **Gustos ocultos** (factorización): el modelo **descubre solo** géneros como «acción» y «romance» y reduce el error cerca de un 40 % (≈ 1.15 → 0.66) en este mundo inventado.
- ✅ **Arranque en frío:** sin historial se recomienda lo popular; con tres calificaciones ya se puede personalizar. Pero lo popular **se refuerza a sí mismo**.

> 🎓 **Siguiente paso:** [el cuaderno estándar](../03_Sistemas_de_Recomendacion.ipynb) hace todo **desde cero y verificado**, con métricas de ranking, popularidad, ítems nuevos y una sorpresa: el modelo con menor error **no siempre** recomienda mejor según los datos observados. Y para cerrar el módulo, [el triage de urgencias](00_Casos_Estudio_Dummies.ipynb).

---
##### 🛠️ Práctica 1: ¿Qué nota pondría Diego a «Acción 1»?

Repitamos la receta del amigo más parecido, pero ahora para **Diego**, que **no ha visto** «Acción 1». Usa la tabla `T`, las listas `amigos` y `peliculas` y la función `similitud(a, b)` que ya definimos:

1. Calcula el parecido de Diego (fila 3 de `T`) con cada uno de los otros cuatro amigos y guárdalo en un diccionario `sims_d` (clave: el nombre del amigo).
2. Busca la columna de «Acción 1» (`peliculas.index(...)`) y arma la lista `votos_d` con los amigos que **sí vieron** esa película **y** tienen parecido positivo con Diego: tuplas `(nombre, parecido, nota)`.
3. Calcula `prediccion_d`: el promedio de esas notas, dando más peso a los más parecidos (suma de parecido × nota ÷ suma de parecidos).
4. Comprueba que hay al menos un voto y que la predicción es **menor que 3** estrellas (Diego se parece a quienes no aman la acción).

In [ ]:
# Escribe tu código aquí

# diego = T[3]
# sims_d = {amigos[k]: similitud(diego, T[k]) for k in [0, 1, 2, 4]}
# print({k: round(v, 2) for k, v in sims_d.items()})
#
# j = peliculas.index("Acción 1")
# votos_d = [(amigos[k], sims_d[amigos[k]], T[k, j]) for k in [0, 1, 2, 4]
#            if not np.isnan(T[k, j]) and sims_d[amigos[k]] > 0]
# prediccion_d = sum(s * r for _, s, r in votos_d) / sum(s for _, s, _ in votos_d)


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
diego = T[3]
sims_d = {amigos[k]: similitud(diego, T[k]) for k in [0, 1, 2, 4]}      # parecido de Diego con Ana, Beto, Carla y Elena
print("Parecido de cada amigo con Diego:", {k: round(v, 2) for k, v in sims_d.items()})

j = peliculas.index("Acción 1")
votos_d = [(amigos[k], sims_d[amigos[k]], T[k, j]) for k in [0, 1, 2, 4]
           if not np.isnan(T[k, j]) and sims_d[amigos[k]] > 0]          # vieron la película Y se parecen a Diego
print("Votos:", [(n, round(s, 2), int(r)) for n, s, r in votos_d])

prediccion_d = sum(s * r for _, s, r in votos_d) / sum(s for _, s, _ in votos_d)   # promedio ponderado por el parecido
print(f"Predicción para Diego: {prediccion_d:.1f} estrellas")

assert len(votos_d) >= 1 and prediccion_d < 3
print("Diego se parece a quienes no aman la acción, así que la predicción es baja: el amigo parecido decide.")
```
</details>

---


---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿por qué el parecido entre dos personas se calcula **solo con las películas que ambas vieron**? ¿Qué pasaría si dos personas solo coinciden en **una** película?
2. En el experimento, el modelo de gustos ocultos redujo el error de ≈ 1.15 a ≈ 0.66. ¿Por qué crees que esa mejora sería **menor** con datos reales? Piensa en cuántos gustos tiene realmente una persona.
3. Una tienda recomienda **siempre lo más vendido**. ¿Qué ventajas tiene? ¿Qué le pasa a los productos nuevos o poco conocidos con el tiempo? ¿Cómo lo evitarías sin dejar de ofrecer lo que gusta?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>